# Principia V1.3 Tutorial

This notebook demonstrates the standard real-LLM Principia V1.3 workflow.

The core stages are intentionally separate:

1. **Research**: retrieve real works from public metadata sources.
2. **Information extraction**: extract structured features from retrieved works with a real LLM.
3. **Evidence selection**: choose which extracted ideas, principles, takeaways, baselines, benchmarks, and result facts should feed generation.
4. **Generate idea**: synthesize one evidence-grounded idea with a real LLM.
5. **Compare**: check the generated idea against prior ideas extracted from the retrieved works.


## Installation And Kernel Setup

Install Principia before running this notebook:

```bash
pip install principia-ai ipykernel
```

For local source development from the package repository:

```bash
pip install -e . ipykernel
```

For VS Code, create and select a dedicated notebook kernel:

```bash
python -m ipykernel install --user --name principia-v13-python --display-name "Python 3.12 (Principia V1.3)"
```

Then open this notebook and select `Python 3.12 (Principia V1.3)`. If VS Code does not show the named kernel, select the Python interpreter from the virtual environment where `principia-ai` and `ipykernel` were installed.

This tutorial is designed for real LLM execution only.


## 1. Setup

Replace `YOUR_SILICONFLOW_API_KEY` with your own SiliconFlow API key.

The key is passed directly into `pc.siliconflow_config(...)`; no hidden shell environment is required.


In [1]:
import principia as pc
from IPython.display import Markdown, display

API_key = "sk-vrieqyotsliotomztliuuuhqbtflakoawheplketpysfxelh"

EXTRACT_MODEL = "siliconflow:deepseek-ai/DeepSeek-V4-Pro"
IDEA_MODEL = "siliconflow:Qwen/Qwen3.5-397B-A17B"

goal = "provide real-time, calibrated, actionable process quality control for autonomous coding agents operating on large-scale repositories"

pc.__version__


'1.3.0'

Expected output: `1.3.0`.

## 2. Create Workspace

The workspace stores user-facing outputs in visible files and internal state in `.principia/`:

```text
principia_project/
  README.md
  principia_outputs/
    latest/
      idea.md
      result.json
      works.json
    exports/
      <idea_id>/
        idea.md
        result.json
        works.json
  .principia/
    principia.sqlite
    artifacts/
      source_json/
      exports/
      pdfs/
      cache/
```


In [2]:


ws = pc.Workspace(
    #"principia_project",
    "test-project-1",
    llm_config=pc.siliconflow_config(API_key, timeout=420, max_retries=2),
)

ws.path


PosixPath('/Users/zhengqipei/Desktop/ChipFlow/Principia/projects/Principia-v1.3-tutorial/test-project-1')

## 3. Research: Search Real Works

This cell retrieves 50 real works from public metadata sources and stores them in SQLite.

Principia searches OpenAlex, Crossref, and arXiv. When the same work appears as both an arXiv preprint and a peer-reviewed publication, Principia keeps the arXiv link/ID but promotes the peer-reviewed venue metadata.

The progress display is provided by Principia. It includes elapsed time and ETA.


In [3]:

search_progress = pc.notebook_progress("Research search")

works = ws.research.search(
    goal,
    target_count=50,
    callback=search_progress,
)

len(works), works[0].title


### Research search

**Operation:** `research.search`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 10s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** target_count=50, source=arxiv, raw_candidates=199, works=50

(50,
 'SWE-Sharp-Bench: A Reproducible Benchmark for C# Software Engineering Tasks')

## 4. Review Retrieved Works

A `WorkItem` stores metadata such as `id`, `title`, `authors`, `abstract`, `year`, `venue`, `url`, `doi`, `arxiv_id`, and `openalex_id`.


In [4]:
work_rows = [
    [i + 1, w.title, pc.work_review_status(w), w.year or "", w.venue or w.source, w.url]
    for i, w in enumerate(works[:12])
]
display(Markdown(pc.markdown_table(["#", "Title", "Review status", "Year", "Venue/Source", "URL"], work_rows)))


| # | Title | Review status | Year | Venue/Source | URL |
| --- | --- | --- | --- | --- | --- |
| 1 | SWE-Sharp-Bench: A Reproducible Benchmark for C# Software Engineering Tasks | peer-reviewed | 2025 | 2025 2nd IEEE/ACM International Conference on AI-powered Software (AIware) | https://doi.org/10.1109/aiware69974.2025.00039 |
| 2 | Iterative Audit Convergence in LLM-Managed Multi-Agent Systems: A Case Study in Prompt-Engineering Quality Assurance | peer-reviewed | 2026 | Software | https://doi.org/10.3390/software5020026 |
| 3 | Comparative Evaluation of Gemini and DeepSeek for LLM-Generated Code Quality and Architectural Robustness in Backend Software Engineering | peer-reviewed | 2026 | Electronics | https://doi.org/10.3390/electronics15132805 |
| 4 | SWE-QA: Can Language Models Answer Repository-level Code Questions? | peer-reviewed | 2026 | Findings of the Association for Computational Linguistics: ACL 2026 | https://doi.org/10.18653/v1/2026.findings-acl.402 |
| 5 | Trustworthy and Secure LLM-Assisted Code Generation for Enterprise Software Development | peer-reviewed | 2026 | International journal of data science and machine learning | https://doi.org/10.55640/ijdsml-06-01-04 |
| 6 | SGAgent: Suggestion-Guided LLM-Based Multi-Agent Framework for Repository-Level Software Repair | peer-reviewed | 2026 | ACM Transactions on Software Engineering and Methodology | https://doi.org/10.1145/3818617 |
| 7 | Complying with Coding Standards or Retaining Programming Style: A Quality Outlook at Source Code Level | peer-reviewed | 2008 | Journal of Software Engineering and Applications | https://doi.org/10.4236/jsea.2008.11013 |
| 8 | Evaluating the Lifespan of Code Smells using Software Repository Mining | peer-reviewed | 2012 | 2012 16th European Conference on Software Maintenance and Reengineering | https://doi.org/10.1109/csmr.2012.79 |
| 9 | A Multi-repository Approach to Study the Topology of Open Source Bugs Communities: Implications for Software and Code Quality | peer-reviewed | 2013 | International Journal of Computer Theory and Engineering | https://doi.org/10.7763/ijcte.2013.v5.663 |
| 10 | Enhancing SWE Bench with Context Engineering: A Comparative Study Against Prompt Engineering in LLM-Based Software Tasks | peer-reviewed | 2026 | Journal of Information Systems Engineering and Management | https://doi.org/10.52783/jisem.v11i1s.14079 |
| 11 | RAG-SecCode: Retrieval-Augmented Secure Coding Guidance for Enterprise LLM Software Development | peer-reviewed | 2026 | Frontiers in Emerging Artificial Intelligence and Machine Learning | https://doi.org/10.64917/feaiml/volume03issue06-01 |
| 12 | MCTS-Refined CoT: High-Quality Fine-Tuning Data for LLM-Based Repository Issue Resolution | peer-reviewed | 2025 | 2025 40th IEEE/ACM International Conference on Automated Software Engineering (ASE) | https://doi.org/10.1109/ase63991.2025.00154 |

## 5. Information Extraction

This cell extracts structured research features from the top-ranked works using Qwen.

For a tutorial, extracting the top 12 works keeps runtime and cost reasonable while preserving the full 50-work research list. Increase `EXTRACT_COUNT` if you want broader evidence extraction.

The progress display includes ETA and per-item counts. During a provider call, Principia keeps the display alive with heartbeat updates rather than freezing on a fixed percentage.


In [5]:

EXTRACT_COUNT = 20
extract_progress = pc.notebook_progress("Information extraction")

features = ws.research.extract(
    works[:EXTRACT_COUNT],
    model=EXTRACT_MODEL,
    overwrite=False,
    callback=extract_progress,
)

features.counts(), features.model


### Information extraction

**Operation:** `research.extract`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 18m 24s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** work_id=HumanEvo_An_Evolution_Aware_Benchmark_For_More_Realistic_Evaluation_Of_Repository_Level_Code_Gen, current=20, total=20, extracted=20, llm_wait_seconds=8

({'works': 20,
  'ideas': 29,
  'principles': 25,
  'takeaways': 24,
  'baselines': 25,
  'benchmarks': 17,
  'result_facts': 54},
 'siliconflow:deepseek-ai/DeepSeek-V4-Pro')

## 6. Review Extracted Features

`ExtractedFeatures` is a batch object. Each item is a `WorkFeatures` record with these feature buckets:

- `ideas`: prior or existed ideas extracted from the work.
- `principles`: reusable principles or mechanisms.
- `takeaways`: actionable lessons or result messages.
- `baselines`: comparison methods.
- `benchmarks`: datasets, tasks, or evaluation settings.
- `result_facts`: grounded factual results.

Each extracted record has an `id`. You can use those IDs in the next step to select specific evidence for idea generation.


In [6]:

display(Markdown(pc.feature_summary_markdown(features, limit=8)))

| Work ID | Work title | Existed idea | Principle | Takeaway |
| --- | --- | --- | --- | --- |
| SWE_Sharp_Bench_A_Reproducible_Benchmark_For_C_Software_Engineering_Tasks | SWE-Sharp-Bench: A Reproducible Benchmark for C# Software Engineering T... | Introduce a benchmark of 150 issue-resolving tasks from 17 C# repositories to evaluate AI coding agents on C# software... | More complex patches (more hunks, lines, files) decrease the likelihood of an agent resolving an issue, as widespread e... | C# patches in SWE-Sharp-Bench involve more files, hunks, and lines on average compared to Python patches in SWE-Bench V... |
| Iterative_Audit_Convergence_In_LLM_Managed_Multi_Agent_Systems_A_Case_Study_In_Prompt_Engineerin | Iterative Audit Convergence in LLM-Managed Multi-Agent Systems: A Case... | Using LLM agents to iteratively audit the specification surface of multi-agent systems to find consistency defects. | Defect discovery counts may not decrease monotonically due to cascading edits and audit-scope expansion, leading to flu... | Structured, iterative agent-driven auditing can uncover numerous consistency defects in prompt specifications that woul... |
| Comparative_Evaluation_Of_Gemini_And_DeepSeek_For_LLM_Generated_Code_Quality_And_Architectural_R | Comparative Evaluation of Gemini and DeepSeek for LLM-Generated Code Qu... | A systematic evaluation methodology using a sequence of prompts to simulate a typical vibe-coding process for assessing... | Even the best available foundational LLMs may still require expert human supervision when generating code for productio... | Gemini tends to produce more architecturally robust code with design patterns like DTO, leading to decoupled and mainta... |
| SWE_QA_Can_Language_Models_Answer_Repository_Level_Code_Questions | SWE-QA: Can Language Models Answer Repository-level Code Questions? | A repository-level code question answering benchmark constructed from real GitHub issues to capture multi-file, archite... | Real-world software repositories demand understanding beyond isolated code snippets; effective QA must navigate multipl... | Advanced LLMs, especially when augmented with an agentic framework, can address repository-level QA to some extent, but... |
| Trustworthy_And_Secure_LLM_Assisted_Code_Generation_For_Enterprise_Software_Development | Trustworthy and Secure LLM-Assisted Code Generation for Enterprise Soft... | A six-layer governance-oriented framework to ensure LLM-assisted code generation in enterprise is secure, reliable, and... | Enterprise adoption of LLM-assisted coding should prioritize secure-by-design workflows. | To securely deploy LLMs for code generation in enterprise, implement a governance framework with prompt constraints, re... |
| SGAgent_Suggestion_Guided_LLM_Based_Multi_Agent_Framework_For_Repository_Level_Software_Repair | SGAgent: Suggestion-Guided LLM-Based Multi-Agent Framework for Reposito... | Introduce a suggestion step between localization and repair to bridge the reasoning gap in automated software repair. | Jumping directly from bug location to fix without intermediate reasoning leaves a fundamental gap, as understanding the... | Adding an intermediate suggestion agent that analyzes the bug and proposes repair strategies can significantly improve... |
| Complying_With_Coding_Standards_Or_Retaining_Programming_Style_A_Quality_Outlook_At_Source_Code | Complying with Coding Standards or Retaining Programming Style: A Quali... | Software quality assurance at source code level evolves from emphasizing programming style to adopting and enforcing co... | For effective software quality assurance at source code level, organizations should emphasize compliance with coding st... | Software teams should adopt and enforce coding standards to improve source code quality, rather than relying on individ... |
| Evaluating_The_Lifespan_Of_Code_Smells_Using_Software_Repository_Mining | Evaluating the Lifespan of Code Smells using Software Repository Mining |  |  |  |

## 7. Select Evidence For Idea Generation

By default, `pc.select_evidence(features)` selects all extracted feature buckets.

You can narrow the generation input with:

- `kinds=["ideas", "principles", "takeaways"]`
- `work_ids=[...]`
- `feature_ids=[...]`
- `limit_per_kind=...`

The result is an `EvidencePacket`, which is the direct input to `ws.ideas.generate(...)`.


In [ ]:

for _item in features.items:
    for _feat in _item.ideas:
        print(_feat['id'], ':', _feat['core_idea'])


SWE_Sharp_Bench_A_Reproducible_C_Software_Engineering_Benchmark : Introduce a benchmark of 150 issue-resolving tasks from 17 C# repositories to evaluate AI coding agents on C# software engineering tasks, filling a gap in existing benchmarks.
C_Tasks_Are_More_Challenging_For_AI_Agents_Than_Python_Tasks : Identical model-agent configurations solve 70% of Python tasks in SWE-Bench Verified but only 40% of C# tasks in SWE-Sharp-Bench, indicating a significant performance gap likely due to higher patch complexity and language-specific factors.
Automated_Curation_Pipeline_For_NET_Repositories : A pipeline that automatically constructs Docker environments for C# PRs by parsing configuration files to infer dependencies and build targets, handling .NET-specific hurdles like NuGet/MSBuild asset selection and multi-targeting conflicts.
Iterative_Agent_Driven_Auditing_Of_Prompt_Specifications : Using LLM agents to iteratively audit the specification surface of multi-agent systems to find consisten

In [30]:

for _item in features.items:
    for _feat in _item.principles:
        print(_feat['id'], ':', _feat['argument'])

Patch_Complexity_Negatively_Affects_Resolution_Success : More complex patches (more hunks, lines, files) decrease the likelihood of an agent resolving an issue, as widespread edits across multiple locations are more challenging.
Model_Choice_Is_The_Strongest_Predictor_Of_Resolution_Success : Among factors like patch complexity, language, and agent, the choice of model has the strongest influence on whether an instance is resolved.
Language_Specific_Characteristics_Influence_Agent_Performance_Beyond_Patch_Compl : Even after controlling for patch complexity metrics, programming language has a substantial effect on resolution success, with Python being significantly easier than C# and Java.
Non_Monotonic_Convergence_In_Iterative_Auditing : Defect discovery counts may not decrease monotonically due to cascading edits and audit-scope expansion, leading to fluctuations.
Multi_Vendor_Union_Enhances_Defect_Detection : Combining findings from multiple LLM vendors (OpenAI, Anthropic, Google, xAI

In [33]:

for _item in features.items:
    for _feat in _item.takeaways:
        print(_feat['id'], ':', _feat['message'])


C_Software_Engineering_Tasks_Are_More_Complex_Than_Python_Tasks : C# patches in SWE-Sharp-Bench involve more files, hunks, and lines on average compared to Python patches in SWE-Bench Verified, indicating higher complexity.
Current_AI_Agents_Struggle_More_With_C_Than_Python : The best model-agent combination (OpenHands + GPT-5) achieves only 47.3% resolution on SWE-Sharp-Bench, while Python benchmarks see higher rates (e.g., 70% on SWE-Bench Verified).
Model_Upgrades_Yield_Significant_Gains_On_C_Tasks : Newer models (GPT-5, Claude Sonnet 4) substantially outperform older ones (GPT-4o) on SWE-Sharp-Bench, with resolution rates more than tripling.
Iterative_Auditing_Surfaces_Hidden_Defects : Structured, iterative agent-driven auditing can uncover numerous consistency defects in prompt specifications that would otherwise remain hidden.
Expect_Non_Monotonic_Defect_Discovery : Defect counts may fluctuate across audit rounds due to cascading edits and scope expansion, so do not assume steady

In [34]:

selected_evidence = pc.select_evidence(
    features,
    kinds=["ideas", "principles", "takeaways"],
    feature_ids=[
        "Knowledge_Graph_Based_Toolkit_For_Repository_Level_Reasoning",
        "Context_Engineering_For_LLM_Based_Software_Tasks",
        "Iterative_Agent_Driven_Auditing_Of_Prompt_Specifications",
        "Patch_Complexity_Negatively_Affects_Resolution_Success",
        "Non_Monotonic_Convergence_In_Iterative_Auditing",
        "Multi_Vendor_Union_Enhances_Defect_Detection",
        "Human_In_The_Loop_Principle",
        "Adopt_Coding_Standards_For_Better_Code_Quality",
        "Focus_On_Bug_Community_Composition_And_Size"
    ]
)

selected_evidence.counts()


{'works': 7,
 'ideas': 3,
 'principles': 4,
 'takeaways': 2,
 'baselines': 0,
 'benchmarks': 0,
 'result_facts': 0}

## 8. Generate Idea

This cell generates one evidence-grounded idea from `selected_evidence` using DeepSeek.

The generated `Idea` includes thesis, novelty claim, mechanistic design, methodological details, method variants, validation protocol, metrics, risks, assumptions, source evidence, lineage or trace metadata, and generation metadata.


In [35]:

generate_progress = pc.notebook_progress("Idea generation")

idea = ws.ideas.generate(
    selected_evidence,
    user_note=goal,
    mode="scidialect-evo",
    model=IDEA_MODEL,
    callback=generate_progress,
)

idea.title


### Idea generation

**Operation:** `ideas.generate.scidialect_evo`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 21s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** evidence_items=7, llm_wait_seconds=20, idea_id=Dialectical_Quality_Control_Evolutionary_Audit_Convergence_For_Autonomous_Coding_Agents

'Dialectical Quality Control: Evolutionary Audit Convergence for Autonomous Coding Agents'

## 9. Review Generated Idea

`pc.idea_markdown(...)` renders the full idea card, including Methodological Details and LaTeX equations when present.


In [36]:

display(Markdown(pc.idea_markdown(idea)))


## Dialectical Quality Control: Evolutionary Audit Convergence for Autonomous Coding Agents

**ID:** `Dialectical_Quality_Control_Evolutionary_Audit_Convergence_For_Autonomous_Coding_Agents`  
**Mode:** `scidialect_evo`  
**Model:** `siliconflow:Qwen/Qwen3.5-397B-A17B`

**Thesis:** Real-time process quality control for autonomous coding agents can be achieved by implementing a dialectical evolutionary loop where multi-vendor auditor agents iteratively critique patch complexity and context adherence, converging on actionable fixes despite non-monotonic defect discovery patterns.

### Novelty Claim
- This approach uniquely synthesizes the 'Non-Monotonic Convergence' pattern observed in iterative auditing with 'Patch Complexity' constraints, using a 'Multi-Vendor Union' mechanism to dynamically calibrate actionability thresholds for large-scale repository repairs, moving beyond static prompt engineering to evolutionary context engineering.

### Mechanistic Design
- A closed-loop system where a primary 'Repair Agent' generates patches based on repository context, which are then subjected to a 'Dialectical Audit Panel' comprising diverse LLM vendors. The panel scores patches against complexity limits and coding standards. If the union of auditors detects defects or excessive complexity, the patch is rejected, and the feedback is evolved into a refined context prompt for the next generation of the repair attempt.

### Methodological Details

The method employs an evolutionary cycle where audit feedback modifies the context window of the repair agent, explicitly managing the trade-off between patch breadth and resolution success while accounting for non-monotonic convergence in defect detection.

#### Symbols
- 

#### Equations
- **Complexity-Adjusted Success Probability:** $\Pr(\text{Success}|P_t) = \sigma(\beta_0 + \beta_1 \cdot \text{hunks}(P_t) + \beta_2 \cdot \text{lines}(P_t) + \beta_3 \cdot \text{files}(P_t))$ — Logistic regression model derived from SWE-Sharp-Bench evidence, where negative coefficients ($\beta_{1,2,3} < 0$) indicate that increased patch complexity reduces resolution likelihood.
- **Multi-Vendor Defect Union:** $D_t = \bigcup_{v \in V} \text{Findings}(A_v, P_t, C_t)$ — The total defect set is the union of findings from all vendor auditors, leveraging the principle that multi-vendor unions detect all seeded defects where single vendors fail.
- **Evolutionary Context Update:** $C_{t+1} = \text{Augment}(C_t, \text{Feedback}(D_t, K(P_t)))$ — Context engineering update rule where the next iteration's context is augmented by structured feedback from the current audit, addressing the limitations of static prompt engineering.

#### Workflow
1. **Step:** Initialize Repository Context Graph
2. **Step:** Generate Candidate Patch
3. **Step:** Compute Complexity Metrics
4. **Step:** Execute Multi-Vendor Audit
5. **Step:** Aggregate Defect Union
6. **Step:** Evaluate Convergence State
7. **Step:** Evolve Context Prompt
8. **Step:** Iterate or Commit

#### Reliability Checks
- **Step:** Verify patch complexity stays within bounds defined by logistic regression coefficients to ensure high probability of success.
- **Step:** Cross-validate defect detection across at least four distinct LLM vendors to ensure union completeness.
- **Step:** Monitor defect count trajectory for non-monotonic fluctuations to prevent premature termination of the audit loop.
- **Step:** Ensure context augmentation strictly adheres to repository topology and dependency graphs.

### Method Variants
- Static Threshold Variant: Halts generation if complexity metrics exceed fixed bounds before auditing.
- Single-Vendor Baseline: Uses only one auditor model to test the efficacy of the multi-vendor union.
- Prompt-Only Control: Relies on initial prompt engineering without iterative context evolution.

### Derived Principles
- Complexity-Constrained Generation: Agent outputs must be bounded by complexity thresholds derived from empirical regression data.
- Dialectical Audit Convergence: Quality assurance requires iterative, multi-perspective critique where defect counts may fluctuate before resolving.
- Context Evolution over Prompt Staticity: Dynamic augmentation of context based on audit feedback outperforms static prompt engineering for repository-level tasks.
- Union-Based Defect Detection: Reliance on a single model for auditing is insufficient; a union of diverse models is required for comprehensive coverage.

### Why It Might Work
- It directly addresses the negative correlation between patch complexity and success by enforcing early complexity checks. It leverages the proven superiority of multi-vendor unions in finding defects that single models miss. By accepting non-monotonic convergence as a feature rather than a bug, the system avoids false negatives during the iterative refinement of complex repository-level repairs.

### Validation Protocol
- Deploy the framework on the SWE-Sharp-Bench (C#) and SWE-Bench-Lite datasets. Run 50 issues per variant. Measure the resolution rate, the number of audit rounds to convergence, and the final patch complexity. Compare against baselines using the defined metrics.

### Relevant Baselines
- Standard SWE-Agent with GPT-4o (no iterative audit)
- SGAgent with Knowledge Graph but single-vendor audit
- Prompt Engineering only (no context evolution)

### Metrics
- Resolution Success Rate (%)
- Average Patch Complexity (hunks, lines, files)
- Defect Detection Recall (vs. seeded defects)
- Convergence Rounds (count until 0 defects)
- Non-Monotonicity Index (variance in defect counts per round)

### Risks
- Infinite loops due to persistent non-monotonic defect introduction.
- High computational cost from multi-vendor API calls per iteration.
- Context window overflow during aggressive context augmentation.
- Over-correction leading to trivial patches that pass audits but fail functional tests.

### Assumptions
- Patch complexity metrics (hunks, lines, files) are reliable proxies for resolution difficulty across languages.
- Diverse LLM vendors provide sufficiently orthogonal error detection capabilities.
- Repository context can be effectively serialized into a graph structure for agent consumption.
- Human-in-the-loop is available for final validation of converged patches.

### Source Evidence
- **principles / Patch Complexity Negatively Affects Resolution Success:** More complex patches (more hunks, lines, files) decrease the likelihood of an agent resolving an issue, as widespread edits across multiple locations are more challenging.
- **ideas / Iterative Agent-Driven Auditing of Prompt Specifications:** Using LLM agents to iteratively audit the specification surface of multi-agent systems to find consistency defects.
- **principles / Non-Monotonic Convergence in Iterative Auditing:** Defect discovery counts may not decrease monotonically due to cascading edits and audit-scope expansion, leading to fluctuations.
- **principles / Multi-Vendor Union Enhances Defect Detection:** Combining findings from multiple LLM vendors (OpenAI, Anthropic, Google, xAI) yields a union that detects all seeded defects, outperforming any single vendor.
- **ideas / Knowledge Graph-Based Toolkit for Repository-Level Reasoning:** Construct a Knowledge Graph from the target repository to provide global contextual awareness and enhance repository-level reasoning for repair agents.
- **takeaways / Adopt Coding Standards for Better Code Quality:** Software teams should adopt and enforce coding standards to improve source code quality, rather than relying on individual programming styles.
- **takeaways / Focus on bug community composition and size:** To enhance software quality, project maintainers should pay attention to who reports and fixes bugs, and the size of the bug community.
- **ideas / Context Engineering for LLM-Based Software Tasks:** Context engineering enhances LLM performance on software engineering problems by providing ordered enhancement of model input with pertinent history, architecture, and domain-specific data from code repositories.
- **principles / Human-in-the-Loop Principle:** To mitigate technical debt from LLM-generated code, human oversight must be integrated into the development workflow.

### Trace
```json
{
  "rounds": 3,
  "variants": [
    "scidialect_evo_full",
    "feature_gate",
    "anomaly_review"
  ],
  "evidence_items": 7,
  "top_variant": "scidialect_evo_full"
}
```

### Generation Metadata
```json
{
  "mode": "scidialect_evo",
  "model": "siliconflow:Qwen/Qwen3.5-397B-A17B",
  "evidence_counts": {
    "works": 7,
    "ideas": 3,
    "principles": 4,
    "takeaways": 2,
    "baselines": 0,
    "benchmarks": 0,
    "result_facts": 0
  },
  "selected_work_ids": [
    "SWE_Sharp_Bench_A_Reproducible_Benchmark_For_C_Software_Engineering_Tasks",
    "Iterative_Audit_Convergence_In_LLM_Managed_Multi_Agent_Systems_A_Case_Study_In_Prompt_Engineerin",
    "SGAgent_Suggestion_Guided_LLM_Based_Multi_Agent_Framework_For_Repository_Level_Software_Repair",
    "Complying_With_Coding_Standards_Or_Retaining_Programming_Style_A_Quality_Outlook_At_Source_Code",
    "A_Multi_Repository_Approach_To_Study_The_Topology_Of_Open_Source_Bugs_Communities_Implications_F",
    "Enhancing_SWE_Bench_With_Context_Engineering_A_Comparative_Study_Against_Prompt_Engineering_In_L",
    "Faster_Code_Deeper_Debt_A_Multivocal_Literature_Review_On_Technical_Debt_And_Its_Early_Signs_In"
  ]
}
```


## 10. Data Structure Reference

The tables below show the public fields available on the main returned objects. The full API reference is in `docs/api.md`.


In [37]:

display(Markdown("### Idea fields\n" + pc.schema_markdown(pc.Idea)))
display(Markdown("### WorkFeatures fields\n" + pc.schema_markdown(pc.WorkFeatures)))
display(Markdown("### EvidencePacket fields\n" + pc.schema_markdown(pc.EvidencePacket)))


### Idea fields
| Field | Type | Required |
| --- | --- | --- |
| id | <class 'str'> | required |
| title | <class 'str'> | required |
| thesis | <class 'str'> | required |
| mode | Literal['standard', 'calculus', 'scidialect_evo'] | required |
| novelty_claim | <class 'str'> | optional |
| mechanism_design | list[str] | optional |
| methodological_details | dict[str, Any] | optional |
| method_variants | list[str] | optional |
| why_it_might_work | list[str] | optional |
| validation_protocol | list[str] | optional |
| baselines | list[str] | optional |
| metrics | list[str] | optional |
| risks | list[str] | optional |
| assumptions | list[str] | optional |
| derived_principles | list[str] | optional |
| evidence_work_ids | list[str] | optional |
| source_evidence | list[dict[str, Any]] | optional |
| lineage | dict[str, Any] | optional |
| trace | dict[str, Any] | optional |
| generation_metadata | dict[str, Any] | optional |
| model | <class 'str'> | optional |
| run_id | <class 'str'> | optional |
| created_at | <class 'str'> | optional |

### WorkFeatures fields
| Field | Type | Required |
| --- | --- | --- |
| work_id | <class 'str'> | required |
| title | <class 'str'> | required |
| model | <class 'str'> | required |
| ideas | list[dict[str, Any]] | optional |
| principles | list[dict[str, Any]] | optional |
| baselines | list[dict[str, Any]] | optional |
| benchmarks | list[dict[str, Any]] | optional |
| takeaways | list[dict[str, Any]] | optional |
| result_facts | list[dict[str, Any]] | optional |
| source_excerpt_chars | <class 'int'> | optional |
| retained_pdf_path | <class 'str'> | optional |
| skipped | <class 'bool'> | optional |
| extraction_id | <class 'str'> | optional |
| created_at | <class 'str'> | optional |

### EvidencePacket fields
| Field | Type | Required |
| --- | --- | --- |
| query | <class 'str'> | optional |
| features | list[principia.models.WorkFeatures] | optional |
| user_note | <class 'str'> | optional |
| created_at | <class 'str'> | optional |

## 11. Compare Against Prior Ideas

This optional final step compares the generated idea with ideas extracted from the retrieved works.

The comparison progress display also uses heartbeat updates during long provider calls.


In [38]:

compare_progress = pc.notebook_progress("Idea comparison")

comparison = ws.ideas.compare(
    idea,
    features,
    model=IDEA_MODEL,
    callback=compare_progress,
)

len(comparison.rows)


### Idea comparison

**Operation:** `ideas.compare`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 13s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** llm_wait_seconds=12, rows=4

4

## 12. Review Comparison

In [39]:

comparison_rows = [
    [
        row.get("title", ""),
        row.get("mechanistic_similarity", ""),
        row.get("essential_difference", ""),
        row.get("potential_advantage", ""),
    ]
    for row in comparison.rows[:8]
]

display(Markdown(pc.markdown_table(["Prior work", "Similarity", "Difference", "Advantage"], comparison_rows)))


| Prior work | Similarity | Difference | Advantage |
| --- | --- | --- | --- |
| Iterative Agent-Driven Auditing of Prompt Specifications | Both employ a closed-loop mechanism where an auditor agent identifies defects ($D_t$) to trigger a refinement cycle, accepting non-monotonic convergence patterns where defect counts fluctuate before resolving. | The prior idea audits static prompt specifications for consistency defects in a multi-agent system, whereas the generated idea audits dynamic code patches ($P_t$) for complexity ($K(P)$) and context adherence, evolving the repository context vector ($C_t$) rather than the prompt text itself. | By integrating the 'Multi-Vendor Union' ($U$) mechanism with explicit complexity constraints derived from SWE-Sharp-Bench, the generated approach prevents the audit loop from converging on overly complex patches that single-vendor audits might miss or accept erroneously. |
| Knowledge Graph-Based Toolkit for Repository-Level Reasoning | Both mechanisms rely on constructing a structured representation of the repository (Context Graph vs. Knowledge Graph) to provide global awareness to the agent, moving beyond local file context. | The prior idea uses the Knowledge Graph as a static retrieval toolkit for localization and reasoning, while the generated idea treats the context ($C_t$) as a mutable state variable that is iteratively augmented ($C_{t+1} = ext{Augment}(C_t, ext{Feedback})$) based on audit feedback. | The generated 'Evolutionary Context Update' mechanism dynamically filters the Knowledge Graph content based on specific defect feedback ($D_t$), potentially reducing context noise and focusing the repair agent on relevant dependency paths more effectively than static retrieval. |
| Context Engineering for LLM-Based Software Tasks | Both approaches reject static prompt engineering in favor of structured context augmentation, utilizing repository-derived data (history, architecture) to enhance model input. | The prior idea proposes a general methodology for ordering and enhancing context, whereas the generated idea implements a specific 'Dialectical Audit' control loop where the context evolution is strictly gated by a 'Complexity-Adjusted Success Probability' logistic model. | The generated mechanism explicitly quantifies the trade-off between patch breadth and success using empirical coefficients ($eta_{1,2,3}$), providing a mathematical stopping criterion that the general context engineering framework lacks. |
| C# Tasks are More Challenging for AI Agents than Python Tasks | Both recognize the negative correlation between patch complexity (files, hunks, lines) and resolution success, using this relationship to inform agent behavior or evaluation. | The prior idea identifies the complexity-success correlation as an empirical observation explaining performance gaps, while the generated idea operationalizes this finding into an active 'Complexity-Constrained Generation' mechanism that rejects patches exceeding probabilistic thresholds before auditing. | By enforcing complexity bounds proactively rather than observing them post-hoc, the generated approach aims to steer the search space toward simpler, higher-probability solutions, potentially closing the performance gap identified in C# tasks. |

## 13. Local Outputs

Export the staged workflow result to local files.

The canonical export remains under hidden `.principia/artifacts/exports/`, and a visible mirror is written to:

```text
principia_project/principia_outputs/latest/
  idea.md
  result.json
  works.json
```

The exported `idea.md` includes Methodological Details, formulas, validation plan, risks, source evidence, and comparison highlights.

The database counts show what is stored in SQLite. Rerun with `overwrite=True` only when you intentionally want to redo completed LLM extraction work.


In [40]:
export_path = ws.export(
    goal=goal,
    works=works,
    features=features,
    idea=idea,
    comparison=comparison,
)

ws.counts(), export_path, ws.outputs_dir / "latest"


({'works': 50,
  'extractions': 20,
  'ideas': 2,
  'comparisons': 2,
  'runs': 6,
  'run_events': 691},
 PosixPath('/Users/zhengqipei/Desktop/ChipFlow/Principia/projects/Principia-v1.3-tutorial/test-project-1/.principia/artifacts/exports/Dialectical_Quality_Control_Evolutionary_Audit_Convergence_For_Autonomous_Coding_Agents'),
 PosixPath('/Users/zhengqipei/Desktop/ChipFlow/Principia/projects/Principia-v1.3-tutorial/test-project-1/principia_outputs/latest'))